# Semiconductor Annual Report Research Pipeline

This notebook implements the final company-level pipeline for a semiconductor supply-chain
thesis project. It is designed to process annual report PDFs, retrieve relevant passages,
classify them with a controlled coding scheme, and export one Excel workbook per company.

Main design choices in this final version:

- the notebook supports both a `single_company` test workflow and a `full_corpus` workflow
- each company is processed independently from discovery through export
- final outputs are uncapped: the notebook does not suppress relevant passages above a target
- retrieval is broadened to improve recall
- page and chunk quality are filtered before retrieval
- `strategy_category` is normalized to a controlled set of E1-E7 labels or `I-*`
- the original passage text is never rewritten by the LLM

The notebook is heavily commented so that each step is understandable and easy to adapt.


## 1. Project Overview

The final pipeline runs in this order:

1. Discover all PDFs recursively under the base folder.
2. Parse company names and fiscal years from folder and filename patterns.
3. Filter the file set based on the selected run mode.
4. Extract page-level text from each report.
5. Remove repeated headers and footers and exclude noisy navigation pages.
6. Chunk only the clean pages into research-friendly passages.
7. Build or reuse a per-company vector index.
8. Run a broader retrieval query pack inside each company corpus.
9. Deduplicate retrieved chunks before classification.
10. Classify each retrieved chunk with a controlled output schema.
11. Normalize labels and filter irrelevant chunks.
12. Export one Excel workbook per company plus optional combined QA outputs.

Important principle:
The `passage` field always contains the original chunk text from the annual report.
The model is used only for classification, not rewriting.


## 2. Imports and Setup

Install missing dependencies in your local environment before running the notebook.
Typical installs:

```bash
pip install pandas openpyxl pymupdf sentence-transformers faiss-cpu tqdm openai
```

If you prefer a different PDF parser, vector store, or model provider later, the notebook is
organized so those components can be swapped in helper functions instead of rewriting the whole
workflow.


In [20]:
from __future__ import annotations

import json
import logging
import os
import pickle
import re
import textwrap
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
from dataclasses import asdict, dataclass
from hashlib import sha1
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import faiss
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from pandas.api.types import is_string_dtype
from IPython.display import display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)
load_dotenv(Path("/Users/paulkoslowsky/Github/Jakop") / ".env")


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("semiconductor_pipeline")


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 180)


## 3. Configuration

Edit this section first when you want to change how the notebook runs.

The two main modes are:

- `single_company`: recommended for testing the full workflow on one firm across all years
- `full_corpus`: run the same workflow for every company discovered in the corpus

This final notebook writes company-level outputs by default and keeps a combined master table
only for QA and optional cross-company analysis.


In [21]:
CONFIG: Dict[str, Any] = {
    # Main run control
    "run_mode": "full_corpus",  # Options: "single_company", "full_corpus"
    "selected_company": None,
    "selected_years": None,  # Example: [2024, 2025]
    "max_files": None,  # Optional quick debugging limit
    # Input and output paths
    "base_input_folder": Path("Data"),
    "output_root": Path("outputs/final_pipeline"),
    "company_export_folder": Path("outputs/final_pipeline/company_excels"),
    "combined_export_folder": Path("outputs/final_pipeline/combined"),
    "intermediate_root": Path("outputs/final_pipeline/intermediate"),
    "vector_store_root": Path("outputs/final_pipeline/vector_store"),
    # Caching and reruns
    "rebuild_index": False,
    "use_cached_pages": True,
    "use_cached_chunks": True,
    "use_cached_retrieval": True,
    "use_cached_classification": True,
    # Chunking controls
    "chunk_size_chars": 1800,
    "chunk_overlap_chars": 250,
    "min_chunk_chars": 280,
    # Retrieval controls
    "top_k_retrieval": 25,
    "embedding_batch_size": 32,
    # Output behavior
    "write_company_excels": True,
    "export_combined_master": True,
    "low_output_warning_threshold": 50,
    # Quality filters
    "drop_table_of_contents_pages": False,
    "strip_repeated_headers_footers": True,
    "filter_low_quality_chunks": True,
    # Model settings
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "openai_model_name": "gpt-5.4-mini",
    "openai_api_key_env_var": "OPENAI_API_KEY",
    "classification_max_workers": 40,
    "run_classification": True,
}


COMPANY_SCOPE_RULES: Dict[str, Dict[str, str]] = {
    "Samsung": {
        "retrieval_scope_suffix": (
            "Focus only on Samsung DS division semiconductor memory foundry and firm-level "
            "statements that clearly apply to those businesses."
        ),
        "classification_scope": (
            "For Samsung, only keep passages relevant to the DS Division, semiconductor, memory, "
            "or foundry context. Exclude other business units unless the passage clearly refers to "
            "the whole company and is materially relevant to semiconductor supply-chain strategy."
        ),
    },
    "MediaTek": {
        "retrieval_scope_suffix": (
            "Focus only on MediaTek Inc. core semiconductor business and exclude unrelated "
            "subsidiary contexts."
        ),
        "classification_scope": (
            "For MediaTek, only keep passages about MediaTek Inc. itself and its semiconductor "
            "business. Exclude unrelated subsidiary content that does not materially concern the "
            "semiconductor industry."
        ),
    },
}


# Create the main output folders up front.
for output_path in [
    CONFIG["output_root"],
    CONFIG["company_export_folder"],
    CONFIG["combined_export_folder"],
    CONFIG["intermediate_root"],
    CONFIG["vector_store_root"],
]:
    output_path.mkdir(parents=True, exist_ok=True)


print("Active configuration:")
for key, value in CONFIG.items():
    print(f"- {key}: {value}")


Active configuration:
- run_mode: full_corpus
- selected_company: None
- selected_years: None
- max_files: None
- base_input_folder: Data
- output_root: outputs/final_pipeline
- company_export_folder: outputs/final_pipeline/company_excels
- combined_export_folder: outputs/final_pipeline/combined
- intermediate_root: outputs/final_pipeline/intermediate
- vector_store_root: outputs/final_pipeline/vector_store
- rebuild_index: False
- use_cached_pages: True
- use_cached_chunks: True
- use_cached_retrieval: True
- use_cached_classification: True
- chunk_size_chars: 1800
- chunk_overlap_chars: 250
- min_chunk_chars: 280
- top_k_retrieval: 25
- embedding_batch_size: 32
- write_company_excels: True
- export_combined_master: True
- low_output_warning_threshold: 50
- drop_table_of_contents_pages: False
- strip_repeated_headers_footers: True
- filter_low_quality_chunks: True
- embedding_model_name: sentence-transformers/all-MiniLM-L6-v2
- openai_model_name: gpt-5.4-mini
- openai_api_key_env_var:

## 4. Controlled Taxonomy and Retrieval Query Pack

This section defines:

- the controlled `strategy_category` values used in the final export
- the broader retrieval query pack used to improve recall

The LLM is asked to classify into the controlled taxonomy. In the tightened final coding
scheme, `secondary_category` is limited to `0` or the same E1-E7 framework, and `orientation`
is limited to `proactive`, `reactive`, or `descriptive`.


In [22]:
CONTROLLED_STRATEGY_CATEGORIES: List[str] = [
    "E1 Operational Buffers",
    "E2 Footprint Diversification",
    "E3 Supply Option Diversification",
    "E4 Robust Distribution",
    "E5 Product Standardisation",
    "E6 Partner Network Strengthening",
    "E7 SCRM and Visibility",
]

ALLOWED_ORIENTATION_VALUES: List[str] = [
    "proactive",
    "reactive",
    "descriptive",
]

PREFERRED_INDUCTIVE_CODES: List[str] = [
    "I-GovernmentSubsidy",
    "I-ExportControlCompliance",
    "I-MaterialStockpiling",
]

ALLOWED_GEOPOLITICAL_TRIGGERS: List[str] = [
    "COVID-19 pandemic",
    "Russia-Ukraine war",
    "U.S.-China trade tensions",
    "U.S. export controls on China",
    "CHIPS Act and public subsidy programs",
    "Taiwan Strait tensions",
    "Sanctions or entity-list restrictions",
    "Critical-material or energy supply disruption",
]


THEME_TO_CONTROLLED_CATEGORY: Dict[str, str] = {
    "E1 Operational Buffers": "E1 Operational Buffers",
    "E2 Footprint Diversification": "E2 Footprint Diversification",
    "E3 Supply Option Diversification": "E3 Supply Option Diversification",
    "E4 Robust Distribution": "E4 Robust Distribution",
    "E5 Product Standardisation": "E5 Product Standardisation",
    "E6 Partner Network Strengthening": "E6 Partner Network Strengthening",
    "E7 SCRM and Visibility": "E7 SCRM and Visibility",
    "Broad geopolitical context": "E7 SCRM and Visibility",
}


QUERY_PACK: List[Dict[str, str]] = [
    {
        "retrieval_theme": "E1 Operational Buffers",
        "retrieval_query": "E1 broad",
        "query_text": "semiconductor safety stock inventory buffer capacity shortage supply disruption resilience",
    },
    {
        "retrieval_theme": "E1 Operational Buffers",
        "retrieval_query": "E1 capacity hedge",
        "query_text": "semiconductor capacity buffer substrate inventory build contingency buffer demand volatility",
    },
    {
        "retrieval_theme": "E1 Operational Buffers",
        "retrieval_query": "E1 annual report phrasing",
        "query_text": "annual report semiconductor maintain inventory levels increase buffer stock capacity constraints bottlenecks",
    },
    {
        "retrieval_theme": "E2 Footprint Diversification",
        "retrieval_query": "E2 broad",
        "query_text": "semiconductor manufacturing footprint diversification geographic expansion new fab region nearshoring",
    },
    {
        "retrieval_theme": "E2 Footprint Diversification",
        "retrieval_query": "E2 foundry footprint",
        "query_text": "semiconductor fab location capacity expansion site selection region manufacturing network",
    },
    {
        "retrieval_theme": "E2 Footprint Diversification",
        "retrieval_query": "E2 annual report phrasing",
        "query_text": "annual report semiconductor geographically balanced supply chain manufacturing footprint expansion external foundry region",
    },
    {
        "retrieval_theme": "E3 Supply Option Diversification",
        "retrieval_query": "E3 broad",
        "query_text": "semiconductor dual sourcing alternative supplier second source backup supply substrate wafer materials",
    },
    {
        "retrieval_theme": "E3 Supply Option Diversification",
        "retrieval_query": "E3 supplier resilience",
        "query_text": "semiconductor supplier diversification alternate suppliers procurement sourcing redundancy qualified suppliers",
    },
    {
        "retrieval_theme": "E3 Supply Option Diversification",
        "retrieval_query": "E3 annual report phrasing",
        "query_text": "annual report semiconductor qualify alternate suppliers diversify sourcing reduce supplier dependency",
    },
    {
        "retrieval_theme": "E4 Robust Distribution",
        "retrieval_query": "E4 broad",
        "query_text": "semiconductor logistics resilience distribution network freight transportation disruption flexibility channel management",
    },
    {
        "retrieval_theme": "E4 Robust Distribution",
        "retrieval_query": "E4 shipment continuity",
        "query_text": "semiconductor shipment delays freight constraints distribution continuity customer delivery resilience",
    },
    {
        "retrieval_theme": "E4 Robust Distribution",
        "retrieval_query": "E4 annual report phrasing",
        "query_text": "annual report semiconductor logistics distribution channel freight transport disruption risk flexibility",
    },
    {
        "retrieval_theme": "E5 Product Standardisation",
        "retrieval_query": "E5 broad",
        "query_text": "semiconductor product standardisation platform architecture common design SKU rationalisation complexity reduction",
    },
    {
        "retrieval_theme": "E5 Product Standardisation",
        "retrieval_query": "E5 modular design",
        "query_text": "semiconductor platform reuse modular design common components product simplification supply chain complexity",
    },
    {
        "retrieval_theme": "E5 Product Standardisation",
        "retrieval_query": "E5 annual report phrasing",
        "query_text": "annual report semiconductor common platform architecture standardize product portfolio reduce complexity",
    },
    {
        "retrieval_theme": "E6 Partner Network Strengthening",
        "retrieval_query": "E6 broad",
        "query_text": "semiconductor long term supply agreement capacity reservation partnership collaboration foundry customer agreement",
    },
    {
        "retrieval_theme": "E6 Partner Network Strengthening",
        "retrieval_query": "E6 ecosystem",
        "query_text": "semiconductor strategic partnership supplier collaboration customer collaboration foundry ecosystem secure capacity",
    },
    {
        "retrieval_theme": "E6 Partner Network Strengthening",
        "retrieval_query": "E6 annual report phrasing",
        "query_text": "annual report semiconductor long-term agreements partnerships collaborations supply assurance capacity commitments",
    },
    {
        "retrieval_theme": "E7 SCRM and Visibility",
        "retrieval_query": "E7 broad",
        "query_text": "semiconductor supply chain risk management visibility supplier monitoring business continuity contingency planning risk assessment",
    },
    {
        "retrieval_theme": "E7 SCRM and Visibility",
        "retrieval_query": "E7 monitoring and control",
        "query_text": "semiconductor supplier monitoring early warning control tower business continuity contingency planning supply chain visibility",
    },
    {
        "retrieval_theme": "E7 SCRM and Visibility",
        "retrieval_query": "E7 annual report phrasing",
        "query_text": "annual report semiconductor risk management supplier monitoring continuity planning risk assessment supply chain visibility",
    },
    {
        "retrieval_theme": "Broad geopolitical context",
        "retrieval_query": "Broad geopolitical query",
        "query_text": "semiconductor export controls licensing entity list sanctions tariff compliance chips act government subsidies critical materials stockpiling",
    },
    {
        "retrieval_theme": "Broad geopolitical context",
        "retrieval_query": "Broad geopolitical manufacturing",
        "query_text": "semiconductor china export control manufacturing equipment restrictions localization regionalization decoupling subsidy compliance supply chain",
    },
]


EXPECTED_CLASSIFICATION_KEYS: List[str] = [
    "relevant",
    "strategy_category",
    "secondary_category",
    "geopolitical_trigger",
    "orientation",
    "main_point",
    "confidence",
]


## 5. Data Classes and Core Utilities

These data classes keep the main pipeline records explicit and readable.
The utility functions handle common tasks such as slug creation, persistence, and safe text
normalization.


In [23]:
@dataclass
class FileRecord:
    """Metadata for one discovered PDF file."""

    firm_name: Optional[str]
    fiscal_year: Optional[int]
    source_file: str
    folder_name: str
    file_name: str


@dataclass
class PageRecord:
    """Page-level record after extraction and cleaning."""

    firm_name: Optional[str]
    fiscal_year: Optional[int]
    source_file: str
    page_number: int
    section: Optional[str]
    page_text: str
    page_text_raw: str
    is_excluded: bool
    exclusion_reason: Optional[str]
    qa_flags: List[str]


@dataclass
class ChunkRecord:
    """Chunk-level record preserved through retrieval and export."""

    firm_name: Optional[str]
    fiscal_year: Optional[int]
    source_file: str
    page_number: int
    section: Optional[str]
    chunk_id: str
    passage: str
    qa_flags: List[str]
    is_excluded: bool
    exclusion_reason: Optional[str]


def save_pickle(obj: Any, path: Path) -> None:
    """Save a Python object to disk as a pickle file."""

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as file_handle:
        pickle.dump(obj, file_handle)


def load_pickle(path: Path) -> Any:
    """Load a pickle file from disk."""

    with path.open("rb") as file_handle:
        return pickle.load(file_handle)


def slugify(value: str) -> str:
    """Create a stable filesystem- and code-friendly slug."""

    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", value.strip())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned.lower() or "unknown"


def normalise_whitespace(text: str) -> str:
    """Collapse messy whitespace while keeping paragraph breaks readable."""

    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_for_matching(text: str) -> str:
    """Normalize a string for comparisons such as repeated line detection."""

    text = normalise_whitespace(text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()


def soft_normalize_passage(text: str) -> str:
    """Normalize a passage for conservative near-duplicate detection."""

    text = normalize_for_matching(text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def alpha_ratio(text: str) -> float:
    """Return the share of alphabetic characters in a text."""

    if not text:
        return 0.0
    alpha_count = sum(character.isalpha() for character in text)
    return alpha_count / max(len(text), 1)


def build_company_output_paths(company_name: str) -> Dict[str, Path]:
    """Create the set of stable output paths used for one company."""

    company_slug = slugify(company_name)
    intermediate_dir = CONFIG["intermediate_root"] / company_slug
    vector_dir = CONFIG["vector_store_root"] / company_slug
    export_dir = CONFIG["company_export_folder"]

    intermediate_dir.mkdir(parents=True, exist_ok=True)
    vector_dir.mkdir(parents=True, exist_ok=True)
    export_dir.mkdir(parents=True, exist_ok=True)

    return {
        "company_slug": Path(company_slug),
        "intermediate_dir": intermediate_dir,
        "vector_dir": vector_dir,
        "manifest_path": intermediate_dir / "file_manifest.csv",
        "pages_path": intermediate_dir / "pdf_pages.pkl",
        "chunks_path": intermediate_dir / "chunks.pkl",
        "retrieval_path": intermediate_dir / "retrieved_chunks.pkl",
        "classification_path": intermediate_dir / "classified_chunks.pkl",
        "final_path": intermediate_dir / "final_relevant_chunks.pkl",
        "summary_path": intermediate_dir / "company_summary.csv",
        "faiss_index_path": vector_dir / "chunks.faiss",
        "faiss_metadata_path": vector_dir / "chunk_metadata.pkl",
        "company_excel_path": export_dir / f"{company_slug}_evidence.xlsx",
    }


## 6. File Discovery and Metadata Parsing

This section discovers all PDF files recursively and parses metadata from the folder/file
structure.

Important metadata rules:

- filename parsing is preferred over folder parsing
- folder parsing is used as a fallback
- year parsing is flexible but warnings are logged when a year cannot be found
- `single_company` mode filters to the chosen firm across all matching years


In [24]:
def extract_year_from_text(text: str) -> Optional[int]:
    """Extract the first plausible four-digit year from text."""

    match = re.search(r"(19|20)\d{2}", text)
    if not match:
        return None
    return int(match.group(0))


def clean_company_name(raw_name: str) -> str:
    """Normalize company names parsed from folder names or filenames."""

    cleaned = re.sub(r"[_\-]+", " ", raw_name)
    cleaned = re.sub(
        r"\b(annual|report|form 10 k|form 20 f|integrated|halfyear|half year|interim)\b",
        " ",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def parse_firm_and_year_from_path(pdf_path: Path) -> FileRecord:
    """Parse `firm_name` and `fiscal_year` from a PDF path."""

    folder_name = pdf_path.parent.name
    file_name = pdf_path.name
    stem = pdf_path.stem

    fiscal_year = extract_year_from_text(stem)
    if fiscal_year is None:
        fiscal_year = extract_year_from_text(folder_name)

    firm_name_from_filename = clean_company_name(re.sub(r"(19|20)\d{2}", " ", stem))
    firm_name_from_folder = clean_company_name(folder_name)
    firm_name = firm_name_from_filename or firm_name_from_folder or None

    if fiscal_year is None:
        logger.warning("Could not parse fiscal year for %s", pdf_path)

    return FileRecord(
        firm_name=firm_name,
        fiscal_year=fiscal_year,
        source_file=str(pdf_path.resolve()),
        folder_name=folder_name,
        file_name=file_name,
    )


def discover_pdf_files(
    base_folder: Path,
    selected_company: Optional[str] = None,
    selected_years: Optional[List[int]] = None,
    max_files: Optional[int] = None,
) -> pd.DataFrame:
    """Discover all PDF files recursively and return a manifest DataFrame."""

    records: List[FileRecord] = []

    for pdf_path in sorted(base_folder.rglob("*.pdf")):
        record = parse_firm_and_year_from_path(pdf_path)

        if selected_company and (record.firm_name or "").lower() != selected_company.lower():
            continue

        if selected_years and record.fiscal_year not in selected_years:
            continue

        records.append(record)

        if max_files is not None and len(records) >= max_files:
            break

    manifest_df = pd.DataFrame([asdict(record) for record in records])

    if manifest_df.empty:
        logger.warning("No PDF files found for the current selection.")
    else:
        logger.info(
            "Discovered %s PDFs across %s companies",
            len(manifest_df),
            manifest_df["firm_name"].nunique(dropna=True),
        )

    return manifest_df


def select_manifest_for_run(config: Dict[str, Any]) -> pd.DataFrame:
    """Select the manifest subset based on the configured run mode."""

    if config["run_mode"] == "single_company":
        return discover_pdf_files(
            base_folder=config["base_input_folder"],
            selected_company=config["selected_company"],
            selected_years=config["selected_years"],
            max_files=config["max_files"],
        )

    if config["run_mode"] == "full_corpus":
        return discover_pdf_files(
            base_folder=config["base_input_folder"],
            selected_company=None,
            selected_years=config["selected_years"],
            max_files=config["max_files"],
        )

    raise ValueError(f"Unsupported run_mode: {config['run_mode']}")


## 7. Page Extraction, Cleaning, and Page-Level QA

This section improves the earlier extraction logic in three ways:

- it strips repeated headers and footers at the report level
- it identifies probable table-of-contents or navigation pages before chunking
- it keeps page-level QA information so you can inspect what was excluded and why

The goal is to stop noisy content from entering the vector index in the first place.


In [25]:
def normalize_line_key(line: str) -> str:
    """Normalize a line for repeated header/footer detection."""

    line = normalize_for_matching(line)
    line = re.sub(r"\bpage\s+\d+\b", " ", line)
    line = re.sub(r"\b\d+\b", " ", line)
    line = re.sub(r"\s+", " ", line)
    return line.strip()


def is_repeated_line_candidate(line: str) -> bool:
    """Return True if a line is a plausible repeated header/footer candidate."""

    line = normalise_whitespace(line)
    if not line:
        return False
    if len(line) < 3 or len(line) > 120:
        return False
    if alpha_ratio(line) < 0.25:
        return False
    return True


def detect_repeated_edge_lines(page_line_bundles: List[List[str]]) -> set[str]:
    """Detect lines that repeat on many page edges within the same PDF."""

    edge_counter: Counter[str] = Counter()
    total_pages = len(page_line_bundles)
    min_hits = max(3, int(total_pages * 0.20))

    for page_lines in page_line_bundles:
        edge_lines = page_lines[:4] + page_lines[-4:]
        seen_keys = set()
        for line in edge_lines:
            if is_repeated_line_candidate(line):
                seen_keys.add(normalize_line_key(line))
        for line_key in seen_keys:
            edge_counter[line_key] += 1

    return {
        line_key
        for line_key, hit_count in edge_counter.items()
        if hit_count >= min_hits
    }


def clean_page_lines(
    raw_lines: List[str],
    repeated_edge_lines: set[str],
    strip_repeated_headers_footers: bool = True,
) -> List[str]:
    """Remove repeated headers/footers and trivial page-number lines from one page."""

    cleaned_lines = [normalise_whitespace(line) for line in raw_lines if normalise_whitespace(line)]
    page_number_pattern = re.compile(r"^(page\s+)?\d{1,4}$", flags=re.IGNORECASE)

    while cleaned_lines and page_number_pattern.match(cleaned_lines[0].strip()):
        cleaned_lines.pop(0)
    while cleaned_lines and page_number_pattern.match(cleaned_lines[-1].strip()):
        cleaned_lines.pop()

    if strip_repeated_headers_footers:
        lines_changed = True
        while cleaned_lines and lines_changed:
            lines_changed = False
            while cleaned_lines and normalize_line_key(cleaned_lines[0]) in repeated_edge_lines:
                cleaned_lines.pop(0)
                lines_changed = True
            while cleaned_lines and normalize_line_key(cleaned_lines[-1]) in repeated_edge_lines:
                cleaned_lines.pop()
                lines_changed = True
            while cleaned_lines and page_number_pattern.match(cleaned_lines[0].strip()):
                cleaned_lines.pop(0)
                lines_changed = True
            while cleaned_lines and page_number_pattern.match(cleaned_lines[-1].strip()):
                cleaned_lines.pop()
                lines_changed = True

    cleaned_lines = [line for line in cleaned_lines if not page_number_pattern.match(line.strip())]

    return cleaned_lines


def is_probable_table_of_contents(page_lines: List[str], page_text: str) -> bool:
    """Heuristically detect table-of-contents or navigation pages."""

    if not page_lines:
        return True

    top_text = " ".join(page_lines[:8]).lower()
    if "table of contents" in top_text:
        return True

    if page_lines and page_lines[0].strip().lower() == "contents":
        return True

    dot_leader_count = sum(1 for line in page_lines if re.search(r"\.{3,}", line))
    page_number_endings = sum(1 for line in page_lines if re.search(r"\s\d{1,4}$", line))
    short_line_count = sum(1 for line in page_lines if len(line) < 80)

    if dot_leader_count >= 4 and short_line_count >= 6:
        return True

    if page_number_endings >= 8 and short_line_count >= 12:
        return True

    return False


def is_probable_low_information_page(page_text: str) -> bool:
    """Detect pages that are too low-information to be useful for retrieval."""

    if len(page_text) < 120:
        return True
    if alpha_ratio(page_text) < 0.35:
        return True
    return False


def detect_section_heading(page_lines: List[str]) -> Optional[str]:
    """Extract a conservative section heading from the top of a clean page."""

    for line in page_lines[:6]:
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) < 4 or len(stripped) > 120:
            continue
        if "table of contents" in stripped.lower():
            continue
        if re.search(r"\.{3,}", stripped):
            continue
        if re.search(r"\b\d{4}\b", stripped):
            continue
        if stripped.isupper() or re.match(r"^[A-Z][A-Za-z0-9/&,\-\s]{3,}$", stripped):
            return stripped
    return None


def extract_pdf_pages(pdf_path: Path, fallback_record: FileRecord) -> List[PageRecord]:
    """Extract clean page-level text and QA flags from one PDF."""

    page_records: List[PageRecord] = []

    try:
        with fitz.open(pdf_path) as document:
            raw_page_lines: List[List[str]] = []

            for page_index in range(len(document)):
                raw_text = document.load_page(page_index).get_text("text")
                raw_lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
                raw_page_lines.append(raw_lines)

            repeated_edge_lines = detect_repeated_edge_lines(raw_page_lines)

            for page_index, raw_lines in enumerate(raw_page_lines):
                page_text_raw = normalise_whitespace("\n".join(raw_lines))
                cleaned_lines = clean_page_lines(
                    raw_lines=raw_lines,
                    repeated_edge_lines=repeated_edge_lines,
                    strip_repeated_headers_footers=CONFIG["strip_repeated_headers_footers"],
                )
                page_text = normalise_whitespace("\n".join(cleaned_lines))

                qa_flags: List[str] = []
                is_excluded = False
                exclusion_reason: Optional[str] = None

                if not page_text:
                    qa_flags.append("empty_after_cleaning")
                    is_excluded = True
                    exclusion_reason = "empty_after_cleaning"

                if (
                    not is_excluded
                    and CONFIG["drop_table_of_contents_pages"]
                    and is_probable_table_of_contents(cleaned_lines, page_text)
                ):
                    qa_flags.append("table_of_contents")
                    is_excluded = True
                    exclusion_reason = "table_of_contents"

                if not is_excluded and is_probable_low_information_page(page_text):
                    qa_flags.append("low_information_page")
                    is_excluded = True
                    exclusion_reason = "low_information_page"

                section = None if is_excluded else detect_section_heading(cleaned_lines)

                page_records.append(
                    PageRecord(
                        firm_name=fallback_record.firm_name,
                        fiscal_year=fallback_record.fiscal_year,
                        source_file=fallback_record.source_file,
                        page_number=page_index + 1,
                        section=section,
                        page_text=page_text,
                        page_text_raw=page_text_raw,
                        is_excluded=is_excluded,
                        exclusion_reason=exclusion_reason,
                        qa_flags=qa_flags,
                    )
                )
    except Exception as error:  # noqa: BLE001
        logger.warning("Failed to read PDF %s: %s", pdf_path, error)

    return page_records


def build_page_summary(pages_df: pd.DataFrame) -> pd.DataFrame:
    """Build a small page-level QA summary table."""

    if pages_df.empty:
        return pd.DataFrame()

    summary_rows = []
    for firm_name, firm_df in pages_df.groupby("firm_name", dropna=False):
        summary_rows.append(
            {
                "firm_name": firm_name,
                "pages_total": len(firm_df),
                "pages_excluded": int(firm_df["is_excluded"].sum()),
                "pages_kept": int((~firm_df["is_excluded"]).sum()),
                "toc_pages": int((firm_df["exclusion_reason"] == "table_of_contents").sum()),
                "low_info_pages": int((firm_df["exclusion_reason"] == "low_information_page").sum()),
            }
        )
    return pd.DataFrame(summary_rows)


## 8. Chunking and Chunk-Level QA

The new chunking logic is designed to avoid the problems seen in the earlier Intel run:

- chunks should not start mid-word
- overlap should not duplicate the same text awkwardly
- low-information chunks should be filtered before indexing

Chunking happens within each page so traceability remains clear.


In [26]:
def split_text_into_units(paragraph: str) -> List[str]:
    """Split a paragraph into sentence-like units for cleaner chunk boundaries."""

    paragraph = normalise_whitespace(paragraph)
    if not paragraph:
        return []

    sentence_candidates = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9(])", paragraph)
    units = [candidate.strip() for candidate in sentence_candidates if candidate.strip()]

    if len(units) == 1 and len(paragraph) > CONFIG["chunk_size_chars"]:
        # Fallback for very long blocks with poor sentence punctuation.
        fallback_units = re.split(r"(?<=;)\s+|(?<=:)\s+|(?<=,)\s+(?=[A-Z])", paragraph)
        units = [candidate.strip() for candidate in fallback_units if candidate.strip()]

    return units or [paragraph]


def split_page_text_into_units(page_text: str) -> List[str]:
    """Split a clean page into sentence-like units while roughly preserving paragraphs."""

    units: List[str] = []
    paragraphs = [paragraph.strip() for paragraph in re.split(r"\n{2,}", page_text) if paragraph.strip()]

    for paragraph in paragraphs:
        paragraph_units = split_text_into_units(paragraph)
        units.extend(paragraph_units)

    return units


def build_chunks_from_units(
    units: List[str],
    chunk_size_chars: int,
    chunk_overlap_chars: int,
    min_chunk_chars: int,
) -> List[str]:
    """Build overlapping chunks from sentence-like units."""

    if not units:
        return []

    chunks: List[str] = []
    current_units: List[str] = []
    current_length = 0

    for unit in units:
        projected_length = current_length + len(unit) + (1 if current_units else 0)
        if projected_length <= chunk_size_chars:
            current_units.append(unit)
            current_length = projected_length
            continue

        if current_units:
            chunk_text = normalise_whitespace(" ".join(current_units))
            if len(chunk_text) >= min_chunk_chars:
                chunks.append(chunk_text)

        overlap_units: List[str] = []
        overlap_length = 0
        for previous_unit in reversed(current_units):
            additional_length = len(previous_unit) + (1 if overlap_units else 0)
            if overlap_length + additional_length > chunk_overlap_chars:
                break
            overlap_units.insert(0, previous_unit)
            overlap_length += additional_length

        current_units = overlap_units + [unit]
        current_length = len(normalise_whitespace(" ".join(current_units)))

    if current_units:
        chunk_text = normalise_whitespace(" ".join(current_units))
        if len(chunk_text) >= min_chunk_chars:
            chunks.append(chunk_text)

    return chunks


def chunk_quality_flags(chunk_text: str) -> List[str]:
    """Assign QA flags to chunks that look noisy or unhelpful."""

    flags: List[str] = []
    lowered = chunk_text.lower()

    if "table of contents" in lowered:
        flags.append("table_of_contents")

    if alpha_ratio(chunk_text) < 0.45:
        flags.append("low_alpha_ratio")

    if re.search(r"\.{3,}", chunk_text):
        flags.append("dot_leaders")

    if len(chunk_text) < CONFIG["min_chunk_chars"]:
        flags.append("too_short")

    return flags


def build_chunk_id(
    firm_name: Optional[str],
    fiscal_year: Optional[int],
    source_file: str,
    page_number: int,
    passage: str,
) -> str:
    """Create a deterministic chunk identifier from metadata and passage text."""

    raw_value = "||".join(
        [
            str(firm_name or ""),
            str(fiscal_year or ""),
            source_file,
            str(page_number),
            passage,
        ]
    )
    return sha1(raw_value.encode("utf-8")).hexdigest()


def create_chunks_from_pages(pages_df: pd.DataFrame) -> pd.DataFrame:
    """Create chunk records from the clean page DataFrame."""

    chunk_records: List[ChunkRecord] = []

    if pages_df.empty:
        return pd.DataFrame()

    clean_pages_df = pages_df.loc[~pages_df["is_excluded"]].copy()

    for page_row in tqdm(
        clean_pages_df.itertuples(index=False),
        total=len(clean_pages_df),
        desc="Chunking clean pages",
    ):
        units = split_page_text_into_units(page_row.page_text)
        page_chunks = build_chunks_from_units(
            units=units,
            chunk_size_chars=CONFIG["chunk_size_chars"],
            chunk_overlap_chars=CONFIG["chunk_overlap_chars"],
            min_chunk_chars=CONFIG["min_chunk_chars"],
        )

        for chunk_text in page_chunks:
            qa_flags = chunk_quality_flags(chunk_text)
            is_excluded = CONFIG["filter_low_quality_chunks"] and any(
                flag in {"table_of_contents", "low_alpha_ratio", "dot_leaders", "too_short"}
                for flag in qa_flags
            )
            exclusion_reason = qa_flags[0] if is_excluded and qa_flags else None

            chunk_records.append(
                ChunkRecord(
                    firm_name=page_row.firm_name,
                    fiscal_year=page_row.fiscal_year,
                    source_file=page_row.source_file,
                    page_number=page_row.page_number,
                    section=page_row.section,
                    chunk_id=build_chunk_id(
                        firm_name=page_row.firm_name,
                        fiscal_year=page_row.fiscal_year,
                        source_file=page_row.source_file,
                        page_number=page_row.page_number,
                        passage=chunk_text,
                    ),
                    passage=chunk_text,
                    qa_flags=qa_flags,
                    is_excluded=is_excluded,
                    exclusion_reason=exclusion_reason,
                )
            )

    chunks_df = pd.DataFrame([asdict(record) for record in chunk_records])

    if not chunks_df.empty:
        chunks_df = chunks_df.drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)

    return chunks_df


def build_chunk_summary(chunks_df: pd.DataFrame) -> pd.DataFrame:
    """Build a small chunk QA summary table."""

    if chunks_df.empty:
        return pd.DataFrame()

    summary_rows = []
    for firm_name, firm_df in chunks_df.groupby("firm_name", dropna=False):
        summary_rows.append(
            {
                "firm_name": firm_name,
                "chunks_total": len(firm_df),
                "chunks_excluded": int(firm_df["is_excluded"].sum()),
                "chunks_indexed": int((~firm_df["is_excluded"]).sum()),
            }
        )
    return pd.DataFrame(summary_rows)


## 9. Embeddings, Vector Indexing, and Retrieval

Retrieval is broadened in this final version in two ways:

- each theme now has multiple queries rather than just one
- retrieval happens inside each company corpus instead of one mixed global index

This gives better recall while preserving company-specific context and cleaner exports.


In [27]:
def load_embedding_model(model_name: str) -> SentenceTransformer:
    """Load the sentence-transformer model used for chunk and query embeddings."""

    logger.info("Loading embedding model: %s", model_name)
    return SentenceTransformer(model_name)


def embed_texts(
    texts: Sequence[str],
    embedding_model: SentenceTransformer,
    batch_size: int,
) -> np.ndarray:
    """Embed texts as normalized float32 vectors for FAISS."""

    embeddings = embedding_model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype("float32")


def save_faiss_index(index: faiss.Index, index_path: Path) -> None:
    """Persist a FAISS index to disk."""

    faiss.write_index(index, str(index_path))


def load_faiss_index(index_path: Path) -> faiss.Index:
    """Load a FAISS index from disk."""

    return faiss.read_index(str(index_path))


def build_or_load_vector_store(
    company_name: str,
    chunks_df: pd.DataFrame,
    embedding_model: SentenceTransformer,
    output_paths: Dict[str, Path],
) -> Tuple[faiss.Index, pd.DataFrame]:
    """Build or load the per-company vector store."""

    index_path = output_paths["faiss_index_path"]
    metadata_path = output_paths["faiss_metadata_path"]

    if (
        not CONFIG["rebuild_index"]
        and index_path.exists()
        and metadata_path.exists()
    ):
        logger.info("Loading existing vector store for %s", company_name)
        return load_faiss_index(index_path), load_pickle(metadata_path)

    indexable_chunks = chunks_df.loc[~chunks_df["is_excluded"]].reset_index(drop=True)
    if indexable_chunks.empty:
        raise ValueError(f"No clean chunks available to index for {company_name}.")

    embeddings = embed_texts(
        texts=indexable_chunks["passage"].tolist(),
        embedding_model=embedding_model,
        batch_size=CONFIG["embedding_batch_size"],
    )

    vector_dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(vector_dimension)
    index.add(embeddings)

    save_faiss_index(index, index_path)
    save_pickle(indexable_chunks, metadata_path)

    return index, indexable_chunks


def build_company_query_pack(company_name: str) -> List[Dict[str, str]]:
    """Build the query pack for one company, including optional scope hints."""

    scope_suffix = COMPANY_SCOPE_RULES.get(company_name, {}).get("retrieval_scope_suffix", "").strip()
    company_queries: List[Dict[str, str]] = []

    for query_row in QUERY_PACK:
        query_text = query_row["query_text"]
        if scope_suffix:
            query_text = f"{query_text}. {scope_suffix}"

        company_queries.append(
            {
                "retrieval_theme": query_row["retrieval_theme"],
                "retrieval_query": query_row["retrieval_query"],
                "query_text": query_text,
            }
        )

    return company_queries


def retrieve_top_k_chunks(
    query_pack: List[Dict[str, str]],
    embedding_model: SentenceTransformer,
    vector_index: faiss.Index,
    chunk_metadata_df: pd.DataFrame,
) -> pd.DataFrame:
    """Run the broader query pack against the company vector index."""

    retrieval_rows: List[Dict[str, Any]] = []

    for query_row in tqdm(query_pack, desc="Running retrieval queries"):
        query_vector = embed_texts(
            texts=[query_row["query_text"]],
            embedding_model=embedding_model,
            batch_size=1,
        )
        scores, indices = vector_index.search(query_vector, CONFIG["top_k_retrieval"])

        for rank, (score, chunk_index) in enumerate(zip(scores[0], indices[0]), start=1):
            if chunk_index < 0:
                continue

            chunk_row = chunk_metadata_df.iloc[int(chunk_index)].to_dict()
            retrieval_rows.append(
                {
                    **chunk_row,
                    "retrieval_theme": query_row["retrieval_theme"],
                    "retrieval_query": query_row["retrieval_query"],
                    "retrieval_query_text": query_row["query_text"],
                    "retrieval_rank": rank,
                    "retrieval_score": float(score),
                    "passage_norm_hash": sha1(
                        soft_normalize_passage(chunk_row["passage"]).encode("utf-8")
                    ).hexdigest(),
                }
            )

    return pd.DataFrame(retrieval_rows)


def deduplicate_retrieved_chunks(retrieval_df: pd.DataFrame) -> pd.DataFrame:
    """Deduplicate retrieval results before classification."""

    if retrieval_df.empty:
        return retrieval_df.copy()

    deduped_df = (
        retrieval_df.sort_values(
            by=["retrieval_score", "retrieval_rank"],
            ascending=[False, True],
        )
        .drop_duplicates(subset=["chunk_id"], keep="first")
        .drop_duplicates(subset=["passage_norm_hash"], keep="first")
        .reset_index(drop=True)
    )

    return deduped_df


## 10. LLM Classification and Label Normalization

The model is used only as a classifier. The passage text is not rewritten.

The final `strategy_category` must be:

- one of the exact E1-E7 controlled labels, or
- an `I-*` inductive code if the passage is relevant but does not fit the framework

A post-classification normalization step enforces this rule even if the raw LLM output drifts.


In [28]:
def get_openai_client(api_key_env_var: str) -> OpenAI:
    """Create an OpenAI client from an environment variable."""

    api_key = os.getenv(api_key_env_var)
    if not api_key:
        raise EnvironmentError(
            f"Environment variable '{api_key_env_var}' is not set. "
            "Set your API key before running classification."
        )
    return OpenAI(api_key=api_key)


_thread_local_state = threading.local()


def get_thread_local_openai_client(api_key_env_var: str) -> OpenAI:
    """Return one OpenAI client per worker thread for safer concurrent requests."""

    client = getattr(_thread_local_state, "openai_client", None)
    client_env = getattr(_thread_local_state, "api_key_env_var", None)

    if client is None or client_env != api_key_env_var:
        client = get_openai_client(api_key_env_var)
        _thread_local_state.openai_client = client
        _thread_local_state.api_key_env_var = api_key_env_var
        logger.info(
            "Initialized OpenAI client in worker thread %s",
            threading.current_thread().name,
        )

    return client


def inductive_code(raw_label: Optional[str]) -> str:
    """Convert a raw label into a valid `I-*` inductive code."""

    raw_label = normalise_whitespace(str(raw_label or "uncategorised"))
    raw_label = re.sub(r"(?i)^i[\s\-_]*", "", raw_label).strip()
    preferred_code = match_preferred_inductive_code(raw_label)
    if preferred_code:
        return preferred_code
    raw_slug = slugify(raw_label).replace("_", "-")
    return f"I-{raw_slug or 'uncategorised'}"


def match_preferred_inductive_code(raw_value: Optional[str]) -> Optional[str]:
    """Map common inductive themes to preferred exact `I-*` labels."""

    if raw_value is None:
        return None

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")
    exact_map = {
        normalize_for_matching(code).replace("_", " "): code
        for code in PREFERRED_INDUCTIVE_CODES
    }
    if lowered in exact_map:
        return exact_map[lowered]

    pattern_map: List[Tuple[List[str], str]] = [
        (["subsidy", "grant", "government support", "government incentive", "public incentive", "public funding", "state aid", "chips act", "industrial policy", "tax credit"], "I-GovernmentSubsidy"),
        (["export control", "export compliance", "licensing", "entity list", "sanction", "trade compliance", "trade restriction", "government restriction"], "I-ExportControlCompliance"),
        (["stockpil", "strategic reserve", "material reserve", "raw material reserve", "critical material inventory", "inventory reserve", "buffer of critical materials"], "I-MaterialStockpiling"),
    ]

    for keywords, code in pattern_map:
        if any(keyword in lowered for keyword in keywords):
            return code

    return None


def normalize_strategy_category(
    raw_value: Optional[str],
    retrieval_theme: Optional[str],
) -> str:
    """Normalize raw model output into the controlled category set."""

    if not raw_value:
        fallback_category = THEME_TO_CONTROLLED_CATEGORY.get(retrieval_theme or "", "E7 SCRM and Visibility")
        return fallback_category

    raw_text = str(raw_value)
    lowered = normalize_for_matching(raw_text).replace("_", " ")

    exact_map = {
        normalize_for_matching(category).replace("_", " "): category
        for category in CONTROLLED_STRATEGY_CATEGORIES
    }

    if lowered in exact_map:
        return exact_map[lowered]

    preferred_inductive_code = match_preferred_inductive_code(raw_value)
    if preferred_inductive_code:
        return preferred_inductive_code

    pattern_map: List[Tuple[List[str], str]] = [
        (["operational buffer", "inventory", "safety stock", "buffer stock", "capacity buffer"], "E1 Operational Buffers"),
        (["footprint", "geographic diversification", "manufacturing expansion", "fab location", "nearshoring", "regional diversification"], "E2 Footprint Diversification"),
        (["dual sourcing", "alternative supplier", "second source", "backup supply", "supplier diversification", "sourcing"], "E3 Supply Option Diversification"),
        (["distribution", "logistics", "freight", "shipment", "channel management", "transport"], "E4 Robust Distribution"),
        (["standardisation", "standardization", "sku", "common design", "platform architecture", "modular design"], "E5 Product Standardisation"),
        (["partnership", "collaboration", "long term supply", "capacity reservation", "ecosystem", "foundry collaboration"], "E6 Partner Network Strengthening"),
        (["risk management", "supplier monitoring", "risk monitoring", "supply chain visibility", "business continuity", "contingency planning", "risk assessment", "control tower", "early warning", "scenario planning", "continuity planning"], "E7 SCRM and Visibility"),
    ]

    for keywords, normalized_category in pattern_map:
        if any(keyword in lowered for keyword in keywords):
            return normalized_category

    if re.match(r"(?i)^i[\s\-_]", raw_text.strip()):
        return inductive_code(raw_text)

    return inductive_code(raw_value)


def normalize_secondary_category(
    raw_value: Optional[str],
    retrieval_theme: Optional[str],
) -> str:
    """Normalize `secondary_category` to either 0 or the same E1-E7 framework."""

    if raw_value is None:
        return "0"

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")
    if lowered in {"0", "none", "null", "na", "n a", "missing", "other", "n/a"}:
        return "0"

    exact_map = {
        normalize_for_matching(category).replace("_", " "): category
        for category in CONTROLLED_STRATEGY_CATEGORIES
    }
    if lowered in exact_map:
        return exact_map[lowered]

    for category in CONTROLLED_STRATEGY_CATEGORIES:
        normalized_category = normalize_strategy_category(raw_value, retrieval_theme)
        if normalized_category == category:
            return normalized_category

    return "0"


def normalize_orientation(raw_value: Optional[str]) -> str:
    """Normalize `orientation` to proactive, reactive, or descriptive."""

    if raw_value is None:
        return "descriptive"

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")

    proactive_keywords = [
        "proactive", "anticipatory", "preventive", "preparedness", "preemptive", "forward looking", "long term"
    ]
    reactive_keywords = [
        "reactive", "response", "responding", "mitigation after", "contingency response", "crisis response"
    ]
    descriptive_keywords = [
        "descriptive", "observational", "context", "background", "narrative", "general", "informational"
    ]

    if any(keyword in lowered for keyword in proactive_keywords):
        return "proactive"
    if any(keyword in lowered for keyword in reactive_keywords):
        return "reactive"
    if any(keyword in lowered for keyword in descriptive_keywords):
        return "descriptive"

    return "descriptive"


def normalize_geopolitical_trigger(raw_value: Optional[str]) -> str:
    """Normalize geopolitical triggers to a small controlled list or blank."""

    if raw_value is None:
        return ""

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")
    if lowered in {"", "0", "none", "null", "na", "n a", "missing", "other", "n/a"}:
        return ""

    exact_map = {
        normalize_for_matching(trigger).replace("_", " "): trigger
        for trigger in ALLOWED_GEOPOLITICAL_TRIGGERS
    }
    if lowered in exact_map:
        return exact_map[lowered]

    pattern_map: List[Tuple[List[str], str]] = [
        (["covid", "pandemic", "coronavirus"], "COVID-19 pandemic"),
        (["ukraine", "russia", "russian invasion"], "Russia-Ukraine war"),
        (["us china trade", "u s china trade", "trade war", "tariff", "u s china tension"], "U.S.-China trade tensions"),
        (["export control", "chip restriction", "china restriction", "october 2022 controls", "technology restriction"], "U.S. export controls on China"),
        (["chips act", "public subsidy", "government subsidy", "state aid", "grant", "incentive"], "CHIPS Act and public subsidy programs"),
        (["taiwan strait", "cross strait", "taiwan tension"], "Taiwan Strait tensions"),
        (["entity list", "sanction", "blacklist", "license requirement"], "Sanctions or entity-list restrictions"),
        (["energy crisis", "critical material", "raw material", "neon", "gas shortage", "material disruption"], "Critical-material or energy supply disruption"),
    ]

    for keywords, trigger in pattern_map:
        if any(keyword in lowered for keyword in keywords):
            return trigger

    return ""


def build_classification_prompt(passage_row: pd.Series) -> str:
    """Build the final classification prompt used for one retrieved chunk."""

    company_scope_text = COMPANY_SCOPE_RULES.get(
        str(passage_row.get("firm_name", "")),
        {},
    ).get("classification_scope", "")

    allowed_categories_text = "\n".join(f"- {category}" for category in CONTROLLED_STRATEGY_CATEGORIES)
    preferred_inductive_text = "\n".join(f"- {code}" for code in PREFERRED_INDUCTIVE_CODES)
    allowed_triggers_text = "\n".join(f"- {trigger}" for trigger in ALLOWED_GEOPOLITICAL_TRIGGERS)

    instructions = textwrap.dedent(
        f"""
        You are classifying an original annual-report passage for a semiconductor supply-chain thesis.

        Important rules:
        1. Do not rewrite, summarize, shorten, or paraphrase the passage.
        2. Your task is classification only.
        3. Return exactly one JSON object and nothing else.
        4. The JSON object must contain exactly these keys:
           relevant
           strategy_category
           secondary_category
           geopolitical_trigger
           orientation
           main_point
           confidence
        5. If relevant is true, `strategy_category` must be either:
           - one of these exact labels:
        {allowed_categories_text}
           - or, if the passage is relevant but does not fit E1-E7, prefer one of these exact inductive labels first:
        {preferred_inductive_text}
           - if none of those fit, use another inductive code formatted as I-<slug>
        6. `secondary_category` must be either:
           - `0`
           - or one of the exact E1-E7 labels above
        7. `orientation` must be exactly one of:
           - proactive
           - reactive
           - descriptive
        8. `geopolitical_trigger` must be either an empty string if no specific trigger is clearly identifiable,
           or one of these exact labels:
        {allowed_triggers_text}
        9. If the passage is not materially relevant to semiconductor supply-chain strategy,
           resilience, sourcing, manufacturing footprint, logistics, partnerships, or risk
           monitoring, return relevant=false.
        10. Use E7 SCRM and Visibility only when the passage clearly describes monitoring,
            visibility, business continuity, contingency planning, early warning, or formal
            supply-chain risk management mechanisms.
        11. Do not use E7 only because a passage mentions geopolitics, trade restrictions,
            regulation, tariffs, or government policy.
        12. Prefer I-GovernmentSubsidy for grants, subsidies, tax credits, public incentives,
            CHIPS-type support, or industrial-policy funding.
        13. Prefer I-ExportControlCompliance for export controls, licensing restrictions,
            entity-list rules, sanctions, and compliance with government trade restrictions.
        14. Prefer I-MaterialStockpiling for strategic stockpiling or reserve-building of
            critical materials rather than routine working inventory.
        15. `confidence` must be numeric between 0 and 1.
        16. Use null only when a field is truly not inferable, but prefer the constrained values
            above wherever possible.
        """
    ).strip()

    metadata_block = textwrap.dedent(
        f"""
        Metadata:
        - firm_name: {passage_row.get("firm_name")}
        - fiscal_year: {passage_row.get("fiscal_year")}
        - source_file: {passage_row.get("source_file")}
        - page_number: {passage_row.get("page_number")}
        - section: {passage_row.get("section")}
        - retrieval_theme: {passage_row.get("retrieval_theme")}
        - retrieval_query: {passage_row.get("retrieval_query")}
        """
    ).strip()

    scope_block = f"Company-specific scope rule:\n{company_scope_text}" if company_scope_text else ""

    return f"{instructions}\n\n{metadata_block}\n\n{scope_block}\n\nPassage:\n{passage_row.get('passage')}"


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Extract the first JSON object from a text response."""

    text = text.strip()
    if not text:
        return None

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def validate_classification_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Validate the raw classification payload before normalization."""

    if set(payload.keys()) != set(EXPECTED_CLASSIFICATION_KEYS):
        raise ValueError(
            f"Invalid JSON keys. Expected exactly {EXPECTED_CLASSIFICATION_KEYS}, got {list(payload.keys())}"
        )

    if not isinstance(payload["relevant"], bool):
        raise ValueError("'relevant' must be a boolean.")

    if payload["orientation"] is not None and not isinstance(payload["orientation"], str):
        raise ValueError("'orientation' must be a string or null.")

    if payload["secondary_category"] is not None and not isinstance(payload["secondary_category"], str):
        raise ValueError("'secondary_category' must be a string or null.")

    if payload["geopolitical_trigger"] is not None and not isinstance(payload["geopolitical_trigger"], str):
        raise ValueError("'geopolitical_trigger' must be a string or null.")

    if not isinstance(payload["confidence"], (int, float)):
        raise ValueError("'confidence' must be numeric.")

    confidence_value = float(payload["confidence"])
    if not 0 <= confidence_value <= 1:
        raise ValueError("'confidence' must be between 0 and 1.")

    return {
        "relevant": payload["relevant"],
        "strategy_category": payload["strategy_category"],
        "secondary_category": payload["secondary_category"],
        "geopolitical_trigger": payload["geopolitical_trigger"],
        "orientation": payload["orientation"],
        "main_point": payload["main_point"],
        "confidence": confidence_value,
    }


def request_openai_json_response(client: OpenAI, model_name: str, prompt: str) -> str:
    """Request a raw model response from OpenAI."""

    response = client.responses.create(
        model=model_name,
        input=prompt,
    )
    return response.output_text


def classify_chunk_with_retries(
    passage_row: pd.Series,
    client: OpenAI,
    model_name: str,
    max_attempts: int = 2,
) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """Classify one chunk and retry once with a repair prompt if needed."""

    prompt = build_classification_prompt(passage_row)
    raw_response_text = ""

    for attempt_number in range(1, max_attempts + 1):
        try:
            if attempt_number == 1:
                raw_response_text = request_openai_json_response(client, model_name, prompt)
            else:
                repair_prompt = textwrap.dedent(
                    f"""
                    The following output was invalid. Repair it into valid JSON only.

                    Required keys:
                    {EXPECTED_CLASSIFICATION_KEYS}

                    Rules:
                    - return exactly one JSON object
                    - do not add markdown or commentary
                    - preserve the original classification intent if possible
                    - relevant must be boolean
                    - confidence must be numeric between 0 and 1
                    - orientation must be proactive, reactive, or descriptive
                    - secondary_category must be 0 or one of the exact E1-E7 labels
                    - geopolitical_trigger must be an empty string or one of the exact allowed trigger labels

                    Invalid output:
                    {raw_response_text}
                    """
                ).strip()
                raw_response_text = request_openai_json_response(client, model_name, repair_prompt)

            parsed_payload = extract_json_object(raw_response_text)
            if parsed_payload is None:
                raise ValueError("No valid JSON object found in the model response.")

            return validate_classification_payload(parsed_payload), None

        except Exception as error:  # noqa: BLE001
            if attempt_number == max_attempts:
                return None, f"Classification failed after {max_attempts} attempts: {error}"

    return None, "Unexpected classification failure."


def classify_chunk_record(
    row_dict: Dict[str, Any],
    model_name: str,
    api_key_env_var: str,
) -> Dict[str, Any]:
    """Classify one retrieved chunk record inside a worker thread."""

    row_series = pd.Series(row_dict)
    client = get_thread_local_openai_client(api_key_env_var)
    payload, failure_message = classify_chunk_with_retries(
        passage_row=row_series,
        client=client,
        model_name=model_name,
    )

    base_record = row_series.to_dict()

    if payload is None:
        logger.warning(
            "Classification failed for chunk %s: %s",
            base_record["chunk_id"],
            failure_message,
        )
        return {
            **base_record,
            "classification_failed": True,
            "classification_error": failure_message,
            "relevant": None,
            "strategy_category_raw": None,
            "strategy_category": None,
            "secondary_category": None,
            "geopolitical_trigger": None,
            "orientation": None,
            "main_point": None,
            "confidence": None,
        }

    normalized_category = (
        normalize_strategy_category(payload["strategy_category"], row_series.get("retrieval_theme"))
        if payload["relevant"]
        else None
    )
    normalized_secondary_category = (
        normalize_secondary_category(payload["secondary_category"], row_series.get("retrieval_theme"))
        if payload["relevant"]
        else "0"
    )
    normalized_orientation = (
        normalize_orientation(payload["orientation"])
        if payload["relevant"]
        else "descriptive"
    )
    normalized_geopolitical_trigger = (
        normalize_geopolitical_trigger(payload["geopolitical_trigger"])
        if payload["relevant"]
        else ""
    )

    return {
        **base_record,
        "classification_failed": False,
        "classification_error": None,
        "relevant": payload["relevant"],
        "strategy_category_raw": payload["strategy_category"],
        "strategy_category": normalized_category,
        "secondary_category": normalized_secondary_category,
        "geopolitical_trigger": normalized_geopolitical_trigger,
        "orientation": normalized_orientation,
        "main_point": payload["main_point"],
        "confidence": payload["confidence"],
    }


def classify_retrieved_chunks(
    retrieved_df: pd.DataFrame,
    model_name: str,
    api_key_env_var: str,
    max_workers: int,
) -> pd.DataFrame:
    """Classify deduplicated retrieved chunks with a concurrent worker pool."""

    if retrieved_df.empty:
        return pd.DataFrame(columns=list(retrieved_df.columns) + EXPECTED_CLASSIFICATION_KEYS)

    row_dicts = retrieved_df.to_dict(orient="records")
    indexed_results: List[Tuple[int, Dict[str, Any]]] = []
    progress_log_interval = max(10, min(50, len(row_dicts) // max(max_workers, 1) or 1))

    logger.info(
        "Starting classification pool for %s retrieved chunks with up to %s worker threads.",
        len(row_dicts),
        max_workers,
    )

    with ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix="classifier") as executor:
        future_to_index = {
            executor.submit(
                classify_chunk_record,
                row_dict=row_dict,
                model_name=model_name,
                api_key_env_var=api_key_env_var,
            ): index
            for index, row_dict in enumerate(row_dicts)
        }

        logger.info(
            "Submitted %s classification tasks to the worker pool.",
            len(future_to_index),
        )
        completed_count = 0

        for future in tqdm(
            as_completed(future_to_index),
            total=len(future_to_index),
            desc=f"Classifying retrieved chunks ({max_workers} workers)",
        ):
            index = future_to_index[future]
            try:
                indexed_results.append((index, future.result()))
                completed_count += 1
                if completed_count % progress_log_interval == 0 or completed_count == len(future_to_index):
                    logger.info(
                        "Classification pool progress: %s/%s chunks completed.",
                        completed_count,
                        len(future_to_index),
                    )
            except Exception as error:  # noqa: BLE001
                row_dict = row_dicts[index]
                logger.warning(
                    "Unhandled worker failure for chunk %s: %s",
                    row_dict.get("chunk_id"),
                    error,
                )
                indexed_results.append(
                    (
                        index,
                        {
                            **row_dict,
                            "classification_failed": True,
                            "classification_error": f"Unhandled worker failure: {error}",
                            "relevant": None,
                            "strategy_category_raw": None,
                            "strategy_category": None,
                            "secondary_category": None,
                            "geopolitical_trigger": None,
                            "orientation": None,
                            "main_point": None,
                            "confidence": None,
                        },
                    )
                )
                completed_count += 1

    indexed_results.sort(key=lambda item: item[0])
    classification_rows = [result for _, result in indexed_results]
    logger.info(
        "Completed classification pool for %s chunks.",
        len(classification_rows),
    )
    return pd.DataFrame(classification_rows)


def build_final_export(classified_df: pd.DataFrame) -> pd.DataFrame:
    """Filter to relevant rows and arrange the final export columns."""

    required_columns = [
        "firm_name",
        "fiscal_year",
        "section",
        "page_number",
        "passage",
        "strategy_category",
        "secondary_category",
        "geopolitical_trigger",
        "orientation",
        "main_point",
        "confidence",
        "source_file",
        "chunk_id",
        "retrieval_query",
        "retrieval_rank",
    ]

    if classified_df.empty or "relevant" not in classified_df.columns:
        return pd.DataFrame(columns=required_columns)

    final_df = classified_df.loc[classified_df["relevant"] == True].copy()  # noqa: E712
    final_df = final_df.drop_duplicates(
        subset=["chunk_id", "retrieval_query", "retrieval_rank"]
    ).reset_index(drop=True)

    final_df = final_df.sort_values(
        by=["fiscal_year", "page_number", "retrieval_query"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    return final_df[required_columns]


## 11. Company-Level Orchestration

This section wraps the helper functions into one company-level pipeline so that:

- `single_company` mode and `full_corpus` mode share the same logic
- all intermediate artifacts can be saved per company
- final exports are naturally split into one workbook per company


In [29]:
def safe_value_counts(dataframe: pd.DataFrame, column_name: str) -> pd.Series:
    """Return value counts safely for empty or missing columns."""

    if dataframe.empty or column_name not in dataframe.columns:
        return pd.Series(dtype="int64")
    return dataframe[column_name].fillna("Missing").value_counts(dropna=False)


EXCEL_ILLEGAL_CHARACTER_RE = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")


def _sanitize_for_excel(value: Any) -> Any:
    """Remove characters that Excel/openpyxl cannot write safely."""

    if not isinstance(value, str):
        return value

    cleaned_value = EXCEL_ILLEGAL_CHARACTER_RE.sub("", value)
    return cleaned_value[:32767]


def sanitize_dataframe_for_excel(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with all string-like columns cleaned for openpyxl."""

    sanitized_df = dataframe.copy()
    for column_name in sanitized_df.columns:
        if is_string_dtype(sanitized_df[column_name]):
            sanitized_df[column_name] = sanitized_df[column_name].map(_sanitize_for_excel)
    return sanitized_df


def export_company_excel(final_df: pd.DataFrame, output_path: Path) -> None:
    """Export the final company DataFrame to Excel."""

    output_path.parent.mkdir(parents=True, exist_ok=True)

    sanitized_df = sanitize_dataframe_for_excel(final_df)
    sanitized_df.to_excel(output_path, index=False, engine="openpyxl")


def run_company_pipeline(
    company_name: str,
    company_manifest_df: pd.DataFrame,
    embedding_model: SentenceTransformer,
    openai_client: Optional[OpenAI],
) -> Dict[str, Any]:
    """Run the full end-to-end pipeline for one company."""

    output_paths = build_company_output_paths(company_name)

    company_manifest_df.to_csv(output_paths["manifest_path"], index=False)

    # -------- Page extraction --------
    if output_paths["pages_path"].exists() and CONFIG["use_cached_pages"]:
        pages_df = load_pickle(output_paths["pages_path"])
    else:
        page_records: List[PageRecord] = []
        for file_row in tqdm(
            company_manifest_df.itertuples(index=False),
            total=len(company_manifest_df),
            desc=f"Extracting pages: {company_name}",
        ):
            fallback_record = FileRecord(
                firm_name=file_row.firm_name,
                fiscal_year=file_row.fiscal_year,
                source_file=file_row.source_file,
                folder_name=file_row.folder_name,
                file_name=file_row.file_name,
            )
            page_records.extend(extract_pdf_pages(Path(file_row.source_file), fallback_record))
        pages_df = pd.DataFrame([asdict(record) for record in page_records])
        save_pickle(pages_df, output_paths["pages_path"])

    # -------- Chunking --------
    if output_paths["chunks_path"].exists() and CONFIG["use_cached_chunks"]:
        chunks_df = load_pickle(output_paths["chunks_path"])
    else:
        chunks_df = create_chunks_from_pages(pages_df)
        save_pickle(chunks_df, output_paths["chunks_path"])

    # -------- Retrieval --------
    if output_paths["retrieval_path"].exists() and CONFIG["use_cached_retrieval"]:
        retrieval_df = load_pickle(output_paths["retrieval_path"])
    else:
        vector_index, indexed_chunks_df = build_or_load_vector_store(
            company_name=company_name,
            chunks_df=chunks_df,
            embedding_model=embedding_model,
            output_paths=output_paths,
        )
        retrieval_df = retrieve_top_k_chunks(
            query_pack=build_company_query_pack(company_name),
            embedding_model=embedding_model,
            vector_index=vector_index,
            chunk_metadata_df=indexed_chunks_df,
        )
        save_pickle(retrieval_df, output_paths["retrieval_path"])

    deduplicated_retrieval_df = deduplicate_retrieved_chunks(retrieval_df)

    # -------- Classification --------
    if not CONFIG["run_classification"]:
        classified_df = pd.DataFrame()
    elif output_paths["classification_path"].exists() and CONFIG["use_cached_classification"]:
        classified_df = load_pickle(output_paths["classification_path"])
    else:
        classified_df = classify_retrieved_chunks(
            retrieved_df=deduplicated_retrieval_df,
            model_name=CONFIG["openai_model_name"],
            api_key_env_var=CONFIG["openai_api_key_env_var"],
            max_workers=CONFIG["classification_max_workers"],
        )
        save_pickle(classified_df, output_paths["classification_path"])

    final_df = build_final_export(classified_df) if CONFIG["run_classification"] else pd.DataFrame()
    save_pickle(final_df, output_paths["final_path"])

    if CONFIG["write_company_excels"] and not final_df.empty:
        export_company_excel(final_df, output_paths["company_excel_path"])

    page_summary = build_page_summary(pages_df)
    chunk_summary = build_chunk_summary(chunks_df)

    company_summary = pd.DataFrame(
        [
            {
                "firm_name": company_name,
                "files_processed": len(company_manifest_df),
                "pages_total": 0 if pages_df.empty else len(pages_df),
                "pages_excluded": 0 if pages_df.empty else int(pages_df["is_excluded"].sum()),
                "chunks_total": 0 if chunks_df.empty else len(chunks_df),
                "chunks_indexed": 0 if chunks_df.empty else int((~chunks_df["is_excluded"]).sum()),
                "retrieved_rows": len(retrieval_df),
                "deduplicated_retrieval_rows": len(deduplicated_retrieval_df),
                "classification_rows": len(classified_df),
                "final_relevant_rows": len(final_df),
                "low_output_warning": bool(
                    CONFIG["run_classification"]
                    and len(final_df) < CONFIG["low_output_warning_threshold"]
                ),
            }
        ]
    )
    company_summary.to_csv(output_paths["summary_path"], index=False)

    if CONFIG["run_classification"] and len(final_df) < CONFIG["low_output_warning_threshold"]:
        logger.warning(
            "%s produced only %s final rows, which is below the configured warning threshold of %s.",
            company_name,
            len(final_df),
            CONFIG["low_output_warning_threshold"],
        )
    elif CONFIG["run_classification"] and len(final_df) >= 120:
        logger.info(
            "%s produced %s final rows. This is high-volume but plausible for a dense company corpus.",
            company_name,
            len(final_df),
        )

    return {
        "company_name": company_name,
        "manifest_df": company_manifest_df,
        "pages_df": pages_df,
        "page_summary_df": page_summary,
        "chunks_df": chunks_df,
        "chunk_summary_df": chunk_summary,
        "retrieval_df": retrieval_df,
        "deduplicated_retrieval_df": deduplicated_retrieval_df,
        "classified_df": classified_df,
        "final_df": final_df,
        "company_summary_df": company_summary,
        "output_paths": output_paths,
    }


## 12. Manifest Preview for the Current Run

This cell shows which files will be processed under the current configuration.
In `single_company` mode, this should show one firm across all matching years.


In [30]:
manifest_df = select_manifest_for_run(CONFIG)

print(f"Manifest rows: {len(manifest_df):,}")
if not manifest_df.empty:
    print(f"Companies in current run: {manifest_df['firm_name'].nunique(dropna=True)}")
    display(manifest_df.head(20))
    display(
        manifest_df["firm_name"]
        .value_counts(dropna=False)
        .rename("file_count")
        .to_frame()
    )


2026-04-14 11:53:28,247 | INFO | Discovered 10 PDFs across 10 companies


Manifest rows: 10
Companies in current run: 10


,firm_name,fiscal_year,source_file,folder_name,file_name
0,ASML,2017,/Users/paulkoslowsky/Github/Jakop/Data/ASML/ASML_2017.pdf,ASML,ASML_2017.pdf
1,Elmos,2017,/Users/paulkoslowsky/Github/Jakop/Data/Elmos/Elmos_2017.pdf,Elmos,Elmos_2017.pdf
2,Infineon,2017,/Users/paulkoslowsky/Github/Jakop/Data/Infineon/Infineon_2017.pdf,Infineon,Infineon_2017.pdf
3,Intel,2017,/Users/paulkoslowsky/Github/Jakop/Data/Intel/Intel_2017.pdf,Intel,Intel_2017.pdf
4,MediaTek,2017,/Users/paulkoslowsky/Github/Jakop/Data/Mediatek/MediaTek_2017.pdf,Mediatek,MediaTek_2017.pdf
5,Micron,2017,/Users/paulkoslowsky/Github/Jakop/Data/Micron/Micron_2017.pdf,Micron,Micron_2017.pdf
6,NXP,2017,/Users/paulkoslowsky/Github/Jakop/Data/NXP/NXP_2017.pdf,NXP,NXP_2017.pdf
7,Qualcom,2017,/Users/paulkoslowsky/Github/Jakop/Data/Qualcom/Qualcom_2017.pdf,Qualcom,Qualcom_2017.pdf
8,Samsung,2017,/Users/paulkoslowsky/Github/Jakop/Data/Samsung/Samsung_2017.pdf,Samsung,Samsung_2017.pdf
9,TSMC,2017,/Users/paulkoslowsky/Github/Jakop/Data/TSMC/TSMC_2017.pdf,TSMC,TSMC_2017.pdf


,file_count
firm_name,
ASML,1
Elmos,1
Infineon,1
Intel,1
MediaTek,1
Micron,1
NXP,1
Qualcom,1
Samsung,1


## 13. Execute the Company-Level Pipeline

This cell runs the actual pipeline.

Behavior:

- in `single_company` mode, the notebook runs one firm across all available years
- in `full_corpus` mode, the notebook loops through all discovered companies

The same company-level function is used in both modes so testing and final production remain
aligned.


In [31]:
embedding_model = load_embedding_model(CONFIG["embedding_model_name"])

company_results: List[Dict[str, Any]] = []

for company_name, company_manifest_df in manifest_df.groupby("firm_name", dropna=False):
    if pd.isna(company_name):
        logger.warning("Skipping a manifest group with missing firm_name.")
        continue

    print(f"\nRunning pipeline for: {company_name}")
    result = run_company_pipeline(
        company_name=str(company_name),
        company_manifest_df=company_manifest_df.reset_index(drop=True),
        embedding_model=embedding_model,
        openai_client=None,
    )
    company_results.append(result)

print(f"\nCompleted pipeline runs for {len(company_results)} company/corpus groups.")


2026-04-14 11:53:34,358 | INFO | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


2026-04-14 11:53:34,380 | INFO | No device provided, using mps
2026-04-14 11:53:34,632 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-14 11:53:34,656 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-14 11:53:34,822 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-14 11:53:34,845 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-14 11:53:34,848 | INFO | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-04-14 11


Running pipeline for: ASML


Running retrieval queries: 100%|██████████| 23/23 [00:00<00:00, 54.52it/s]
2026-04-14 11:53:44,360 | INFO | Starting classification pool for 131 retrieved chunks with up to 40 worker threads.
2026-04-14 11:53:44,387 | INFO | Initialized OpenAI client in worker thread classifier_0
2026-04-14 11:53:44,392 | INFO | Initialized OpenAI client in worker thread classifier_7
2026-04-14 11:53:44,393 | INFO | Initialized OpenAI client in worker thread classifier_8
2026-04-14 11:53:44,393 | INFO | Initialized OpenAI client in worker thread classifier_10
2026-04-14 11:53:44,393 | INFO | Initialized OpenAI client in worker thread classifier_6
2026-04-14 11:53:44,393 | INFO | Initialized OpenAI client in worker thread classifier_14
2026-04-14 11:53:44,393 | INFO | Initialized OpenAI client in worker thread classifier_1
2026-04-14 11:53:44,394 | INFO | Initialized OpenAI client in worker thread classifier_2
2026-04-14 11:53:44,394 | INFO | Initialized OpenAI client in worker thread classifier_17
2026


Running pipeline for: Elmos


...


Running pipeline for: Samsung


Running retrieval queries: 100%|██████████| 23/23 [00:00<00:00, 62.80it/s]
2026-04-14 11:57:38,140 | INFO | Starting classification pool for 93 retrieved chunks with up to 40 worker threads.
2026-04-14 11:57:38,150 | INFO | Initialized OpenAI client in worker thread classifier_2
2026-04-14 11:57:38,150 | INFO | Initialized OpenAI client in worker thread classifier_3
2026-04-14 11:57:38,151 | INFO | Initialized OpenAI client in worker thread classifier_0
2026-04-14 11:57:38,151 | INFO | Initialized OpenAI client in worker thread classifier_6
2026-04-14 11:57:38,152 | INFO | Initialized OpenAI client in worker thread classifier_1
2026-04-14 11:57:38,152 | INFO | Initialized OpenAI client in worker thread classifier_4
2026-04-14 11:57:38,156 | INFO | Initialized OpenAI client in worker thread classifier_7
2026-04-14 11:57:38,156 | INFO | Initialized OpenAI client in worker thread classifier_5
2026-04-14 11:57:38,156 | INFO | Initialized OpenAI client in worker thread classifier_10
2026-04


Running pipeline for: TSMC


Running retrieval queries: 100%|██████████| 23/23 [00:00<00:00, 82.17it/s]
2026-04-14 11:57:50,344 | INFO | Starting classification pool for 147 retrieved chunks with up to 40 worker threads.
2026-04-14 11:57:50,352 | INFO | Initialized OpenAI client in worker thread classifier_2
2026-04-14 11:57:50,353 | INFO | Initialized OpenAI client in worker thread classifier_5
2026-04-14 11:57:50,353 | INFO | Initialized OpenAI client in worker thread classifier_0
2026-04-14 11:57:50,354 | INFO | Initialized OpenAI client in worker thread classifier_7
2026-04-14 11:57:50,354 | INFO | Initialized OpenAI client in worker thread classifier_3
2026-04-14 11:57:50,354 | INFO | Initialized OpenAI client in worker thread classifier_8
2026-04-14 11:57:50,354 | INFO | Initialized OpenAI client in worker thread classifier_1
2026-04-14 11:57:50,354 | INFO | Initialized OpenAI client in worker thread classifier_4
2026-04-14 11:57:50,356 | INFO | Initialized OpenAI client in worker thread classifier_6
2026-04


Completed pipeline runs for 10 company/corpus groups.


## 14. Export Combined QA Outputs

The primary deliverables are the company-specific Excel workbooks.
This section also writes optional combined QA outputs so you can inspect the full corpus after
the company-level runs have finished.


In [32]:
company_summary_df = pd.concat(
    [result["company_summary_df"] for result in company_results],
    ignore_index=True,
) if company_results else pd.DataFrame()

combined_final_df = pd.concat(
    [result["final_df"] for result in company_results if not result["final_df"].empty],
    ignore_index=True,
) if company_results else pd.DataFrame()

combined_summary_path = CONFIG["combined_export_folder"] / "company_summary.csv"
company_summary_df.to_csv(combined_summary_path, index=False)

if CONFIG["export_combined_master"] and not combined_final_df.empty:
    combined_master_excel_path = CONFIG["combined_export_folder"] / "combined_master.xlsx"
    combined_master_pickle_path = CONFIG["combined_export_folder"] / "combined_master.pkl"
    sanitize_dataframe_for_excel(combined_final_df).to_excel(combined_master_excel_path, index=False, engine="openpyxl")
    save_pickle(combined_final_df, combined_master_pickle_path)
    print(f"Combined master export written to: {combined_master_excel_path.resolve()}")

print(f"Company summary written to: {combined_summary_path.resolve()}")
display(company_summary_df)


Combined master export written to: /Users/paulkoslowsky/Github/Jakop/outputs/final_pipeline/combined/combined_master.xlsx
Company summary written to: /Users/paulkoslowsky/Github/Jakop/outputs/final_pipeline/combined/company_summary.csv


,firm_name,files_processed,pages_total,pages_excluded,chunks_total,chunks_indexed,retrieved_rows,deduplicated_retrieval_rows,classification_rows,final_relevant_rows,low_output_warning
0,ASML,1,210,23,477,474,575,131,131,73,False
1,Elmos,1,64,0,192,191,575,88,88,26,True
2,Infineon,1,194,2,452,450,575,123,123,64,False
3,Intel,1,185,13,386,385,575,125,125,43,True
4,MediaTek,1,311,26,480,472,575,77,77,28,True
5,Micron,1,119,0,297,297,575,92,92,42,True
6,NXP,1,143,1,425,423,575,113,113,43,True
7,Qualcom,1,148,1,420,419,575,126,126,41,True
8,Samsung,1,271,4,454,428,575,93,93,35,True
9,TSMC,1,276,21,620,594,575,147,147,65,False


## 15. Quality Checks

This section gives the final QA overview requested for the thesis workflow.
It reports:

- files processed
- page exclusions
- chunk counts
- retrieval counts
- final relevant rows
- counts by company
- counts by year
- counts by normalized `strategy_category`

This section is especially useful after a `single_company` test run.


In [33]:
if not company_summary_df.empty:
    print("Company-level QA summary")
    print("------------------------")
    display(company_summary_df)

if not combined_final_df.empty:
    print("\nFinal rows by company")
    display(safe_value_counts(combined_final_df, "firm_name").rename("count").to_frame())

    print("\nFinal rows by year")
    display(safe_value_counts(combined_final_df, "fiscal_year").rename("count").to_frame())

    print("\nFinal rows by strategy_category")
    display(safe_value_counts(combined_final_df, "strategy_category").rename("count").to_frame())

    print("\nRetrieval counts by company")
    retrieval_counts_df = pd.DataFrame(
        [
            {
                "firm_name": result["company_name"],
                "retrieved_rows": len(result["retrieval_df"]),
                "deduplicated_rows": len(result["deduplicated_retrieval_df"]),
            }
            for result in company_results
        ]
    )
    display(retrieval_counts_df)


Company-level QA summary
------------------------


,firm_name,files_processed,pages_total,pages_excluded,chunks_total,chunks_indexed,retrieved_rows,deduplicated_retrieval_rows,classification_rows,final_relevant_rows,low_output_warning
0,ASML,1,210,23,477,474,575,131,131,73,False
1,Elmos,1,64,0,192,191,575,88,88,26,True
2,Infineon,1,194,2,452,450,575,123,123,64,False
3,Intel,1,185,13,386,385,575,125,125,43,True
4,MediaTek,1,311,26,480,472,575,77,77,28,True
5,Micron,1,119,0,297,297,575,92,92,42,True
6,NXP,1,143,1,425,423,575,113,113,43,True
7,Qualcom,1,148,1,420,419,575,126,126,41,True
8,Samsung,1,271,4,454,428,575,93,93,35,True
9,TSMC,1,276,21,620,594,575,147,147,65,False



Final rows by company


,count
firm_name,
ASML,73
TSMC,65
Infineon,64
Intel,43
NXP,43
Micron,42
Qualcom,41
Samsung,35
MediaTek,28



Final rows by year


,count
fiscal_year,
2017,460



Final rows by strategy_category


,count
strategy_category,
E6 Partner Network Strengthening,119
E2 Footprint Diversification,91
E7 SCRM and Visibility,68
E5 Product Standardisation,65
E1 Operational Buffers,41
E3 Supply Option Diversification,39
I-ExportControlCompliance,14
E4 Robust Distribution,13
I-GovernmentSubsidy,9



Retrieval counts by company


,firm_name,retrieved_rows,deduplicated_rows
0,ASML,575,131
1,Elmos,575,88
2,Infineon,575,123
3,Intel,575,125
4,MediaTek,575,77
5,Micron,575,92
6,NXP,575,113
7,Qualcom,575,126
8,Samsung,575,93
9,TSMC,575,147


## 16. One-Company Test Views

This section is designed for the `single_company` workflow.
It surfaces the company-specific diagnostics that are most useful when refining the pipeline:

- page-level exclusions
- chunk-level QA
- retrieval counts by theme/query
- final label counts
- a few sample rows for manual inspection


In [35]:
if CONFIG["run_mode"] == "single_company" and company_results:
    single_result = company_results[0]

    print("Page summary")
    display(single_result["page_summary_df"])

    print("\nChunk summary")
    display(single_result["chunk_summary_df"])

    if not single_result["retrieval_df"].empty:
        print("\nRetrieval counts by theme")
        display(
            single_result["retrieval_df"]
            .groupby("retrieval_theme")
            .size()
            .rename("count")
            .reset_index()
            .sort_values("count", ascending=False)
        )

        print("\nRetrieval counts by query")
        display(
            single_result["retrieval_df"]
            .groupby("retrieval_query")
            .size()
            .rename("count")
            .reset_index()
            .sort_values("count", ascending=False)
        )

    if not single_result["final_df"].empty:
        print("\nFinal normalized strategy-category counts")
        display(
            single_result["final_df"]["strategy_category"]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

        print("\nSample final rows")
        sample_columns = [
            "firm_name",
            "fiscal_year",
            "page_number",
            "section",
            "retrieval_query",
            "strategy_category",
            "secondary_category",
            "main_point",
            "confidence",
            "passage",
        ]
        display(single_result["final_df"][sample_columns].head(10))


## 17. Manual Inspection Helpers

These optional views are useful when you want to inspect likely problem areas manually.
For example:

- pages excluded as table-of-contents or low-information
- chunks excluded before indexing
- raw classification drift before normalization

These checks are especially useful when testing one company before running the full corpus.


In [36]:
if company_results:
    first_result = company_results[0]

    if not first_result["pages_df"].empty:
        print("Excluded pages preview")
        excluded_pages_preview = first_result["pages_df"].loc[
            first_result["pages_df"]["is_excluded"],
            ["firm_name", "fiscal_year", "page_number", "exclusion_reason", "section", "page_text_raw"],
        ].head(10)
        display(excluded_pages_preview)

    if not first_result["chunks_df"].empty:
        print("\nExcluded chunks preview")
        excluded_chunks_preview = first_result["chunks_df"].loc[
            first_result["chunks_df"]["is_excluded"],
            ["firm_name", "fiscal_year", "page_number", "exclusion_reason", "passage"],
        ].head(10)
        display(excluded_chunks_preview)

    if (
        CONFIG["run_classification"]
        and not first_result["classified_df"].empty
        and "strategy_category_raw" in first_result["classified_df"].columns
    ):
        print("\nRaw-to-normalized strategy label preview")
        preview_columns = [
            "firm_name",
            "fiscal_year",
            "retrieval_theme",
            "strategy_category_raw",
            "strategy_category",
            "secondary_category",
        ]
        display(first_result["classified_df"][preview_columns].head(15))


Excluded pages preview


,firm_name,fiscal_year,page_number,exclusion_reason,section,page_text_raw
0,ASML,2017,1,empty_after_cleaning,NaN,
1,ASML,2017,2,empty_after_cleaning,NaN,
3,ASML,2017,4,empty_after_cleaning,NaN,ASML INTEGRATED REPORT 2017
8,ASML,2017,9,empty_after_cleaning,NaN,
9,ASML,2017,10,empty_after_cleaning,NaN,ASML INTEGRATED REPORT 2017\n3
22,ASML,2017,23,empty_after_cleaning,NaN,
29,ASML,2017,30,empty_after_cleaning,NaN,
37,ASML,2017,38,empty_after_cleaning,NaN,
43,ASML,2017,44,empty_after_cleaning,NaN,
47,ASML,2017,48,empty_after_cleaning,NaN,



Excluded chunks preview


,firm_name,fiscal_year,page_number,exclusion_reason,passage
235,ASML,2017,101,low_alpha_ratio,"Fair value differences 3\n—\n—\n27.9\n—\n—\n—\n27.9\nPurchase of treasury shares\n(4.8)\n—\n—\n(400.0)\n—\n—\n(400.0)\nCancellation of treasury shares\n—\n—\n—\n—\n—\n—\n—\n18,..."
394,ASML,2017,167,low_alpha_ratio,"Total direct compensation, pension and other benefits\nThe remuneration of key management personnel, comprising of members of the BoM in 2017, 2016 and 2015 was as follows:\nFi..."
398,ASML,2017,168,low_alpha_ratio,"Nickl\n1/20/2017\nConditional\nNo\n11,629\n114.05\n1/20/2020\n—\n1/20/2022\n1/22/2016\nConditional\nNo\n11,205\n83.60\n1/22/2019\n—\n1/22/2021\n1/23/2015\nConditional\nNo\n10,7..."



Raw-to-normalized strategy label preview


,firm_name,fiscal_year,retrieval_theme,strategy_category_raw,strategy_category,secondary_category
0,ASML,2017,E7 SCRM and Visibility,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E7 SCRM and Visibility
1,ASML,2017,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E7 SCRM and Visibility
2,ASML,2017,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E7 SCRM and Visibility
3,ASML,2017,E3 Supply Option Diversification,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E3 Supply Option Diversification
4,ASML,2017,E7 SCRM and Visibility,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E7 SCRM and Visibility
5,ASML,2017,E3 Supply Option Diversification,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E7 SCRM and Visibility
6,ASML,2017,E6 Partner Network Strengthening,E6 Partner Network Strengthening,E6 Partner Network Strengthening,0
7,ASML,2017,E6 Partner Network Strengthening,NaN,NaN,NaN
8,ASML,2017,E3 Supply Option Diversification,E3 Supply Option Diversification,E3 Supply Option Diversification,0
9,ASML,2017,E1 Operational Buffers,E1 Operational Buffers,E1 Operational Buffers,0


## Notes on Extension

The final notebook is intentionally organized so major parts can be swapped later:

- PDF parser:
  Replace `extract_pdf_pages()` if you want to use `pdfplumber` or OCR.
- Embedding model:
  Change `embedding_model_name` and keep the rest of the pipeline stable.
- Vector store:
  Replace FAISS helpers if you later prefer Chroma or another local index.
- LLM provider:
  Replace `request_openai_json_response()` and keep the validation and normalization logic.

Recommended workflow:

1. Run `single_company` on Intel and inspect the QA views.
2. Test Samsung and MediaTek to confirm the company-specific scope rules.
3. Once the single-company outputs look clean, switch to `full_corpus`.
